# SFT with RL2

## Dataset

In [2]:
import os
import datasets
import torch
from torch.utils.data import Dataset


def load_dataset(data_path):
    # Handle dataset split specification (format: "split@path")
    if "@" in data_path:
        split, data_path = data_path.split("@")
    else:
        split = "train"  # Default to "train" split if not specified
    
    # Get file extension to determine dataset format
    ext = os.path.splitext(data_path)[-1].strip(".")
    if ext in ["json", "jsonl", "csv", "parquet", "arrow"]:
        if ext == "jsonl":  # Treat JSON Lines as JSON for compatibility
            ext = "json"
        return datasets.load_dataset(ext, data_files=data_path, split=split)
    else:
        # Load from Hugging Face Hub dataset by name if no extension matches
        return datasets.load_dataset(data_path, split=split)


def tokenize_messages(
    tokenizer,  # Tokenizer for converting text to tokens
    messages,  # List of chat messages (each with "role" and "content")
    tool=None,  # Optional parameter for tool integration in conversations
    apply_chat_template=True  # Whether to use tokenizer's chat formatting
):
    # states: context tokens (from user/system messages)
    # actions: target tokens (from assistant responses)
    # action_mask: marks which tokens are training targets (1=assistant, 0=context)
    states, actions, action_mask = [], [], []
    
    for idx, message in enumerate(messages):
        # Process assistant messages (model's target outputs)
        if message["role"] == "assistant":
            state = tokenizer.encode(message["content"], add_special_tokens=False)
            actions.extend(state)  # Add assistant tokens to action targets
            action_mask.extend([1]*len(state))  # Mark as training targets
        else:
            # Process user/system messages (context)
            if apply_chat_template:
                # Use tokenizer's chat template for proper multi-turn formatting
                next_states = tokenizer.apply_chat_template(
                    messages[:idx + 1],  # All messages up to current one
                    tool=tool,
                    # Add generation prompt if next message is from assistant
                    add_generation_prompt=idx + 1 < len(messages) and messages[idx + 1]["role"] == "assistant"
                )
                # Ensure tokenization is incremental (previous tokens remain unchanged)
                assert next_states[:len(states)] == states, "Tokenizer must preserve previous message tokens"
                state = next_states[len(states):]  # Only take new tokens from current message
            else:
                state = tokenizer.encode(message["content"], add_special_tokens=False)
            
            actions.extend([0]*len(state))  # No action target for context
            action_mask.extend([0]*len(state))  # Don't train on context tokens

        states.extend(state)  # Build full state sequence

    # Return tokenized data as PyTorch tensors (shifted for language modeling)
    return {
        "states": torch.LongTensor(states[:-1]),  # Input context (all except last token)
        "actions": torch.LongTensor(actions[1:]),  # Target outputs (shifted by 1)
        "action_mask": torch.LongTensor(action_mask[1:]),  # Mask for training focus
        "position_ids": torch.arange(len(states) - 1)  # Positional encoding indices
    }

class BaseDataset(Dataset):
    """Base class for conversational datasets"""
    
    def __init__(self, data_path, tokenizer, max_length):
        self.dataset = load_dataset(data_path)  # Load dataset using our custom loader
        self.tokenizer = tokenizer  # Store tokenizer for processing
        self.max_length = max_length  # Maximum sequence length for truncation/padding

    def __len__(self):
        return len(self.dataset)  # Return number of samples in dataset

```{tip}
`datasets.load_dataset` 不会一次性将数据全部读入内存，而是通过延迟加载和按需读取实现高效处理。
```

In [4]:
class SFTDataset(BaseDataset):
    
    def __getitem__(self, idx):
        
        messages = self.dataset[idx]["messages"]
        ex = tokenize_messages(self.tokenizer, messages)
        return {k: v[:self.max_length] for k, v in ex.items()}
    
    def collate_fn(self, batch):
        return list(batch)

```{tip}
在 PyTorch 的 `DataLoader` 中，`collate_fn` 的核心作用是将多个样本（由 `__getitem__` 返回）整理成一个批次（batch）数据。默认情况下，`DataLoader` 会尝试将样本自动拼接成张量，但当样本结构复杂（如长度可变的列表、字典）时，需要自定义 `collate_fn` 来处理。<br>
上面 collate_fn 的输入和输出：<br>
* 输入：一个列表，包含 `batch_size` 个样本，每个样本是 `__getitem__` 返回的结果（在本例中是字典，如 `{"states": tensor, "actions": tensor, ...}`）。
* 输出：整理后的批次数据（格式由用户定义，本例中直接返回列表）。
```

## Trainer

In [10]:
from omegaconf import OmegaConf  # For managing configuration files (YAML/JSON)
from torch.utils.data import DataLoader
import torch.distributed as dist  # For multi-GPU distributed training
from transformers import get_cosine_schedule_with_warmup
import wandb


class Trainer:
    """Handles training setup: config management, logging, data loading, and scheduler"""
    
    def __init__(self, config):
        OmegaConf.resolve(config)  # Resolve any interpolations in the config
        self.config = config  # Store configuration for later use

        # Only run on the main process (rank 0) in distributed training
        if dist.get_rank() == 0:
            print(OmegaConf.to_yaml(config))  # Print config for verification
            # Initialize W&B unless disabled
            if not config.trainer.disable_wandb:
                wandb.init(
                    project=config.trainer.project,  # W&B project name
                    name=config.trainer.experiment_name,  # Name for this run
                    config=OmegaConf.to_container(config)  # Log config to W&B
                )
            else:
                # Disable W&B logging by replacing log method with no-op
                wandb.log = lambda *args, **kwargs: None

    def prepare_dataloader(self, dataset, batch_size, shuffle):
        """Create a DataLoader for batching and shuffling data"""
        return DataLoader(
            dataset,
            batch_size,
            shuffle=shuffle,  # Shuffle data (typically True for training)
            drop_last=True,  # Discard incomplete batch at end to maintain consistent size
            collate_fn=dataset.collate_fn  # Use dataset's custom batching logic
        )
    
    def prepare_scheduler(self, worker):
        """Set up cosine learning rate scheduler with warmup"""
        # Calculate total training steps (epochs × batches per epoch)
        num_training_steps = self.config.trainer.n_epochs * len(self.dataloader)
        # Calculate warmup stepbalanceds (fraction of total steps defined in config)
        num_warmup_steps = int(worker.config.warmup_ratio * num_training_steps)

        # Return scheduler: starts with warmup, then decays lr cosine-style
        return get_cosine_schedule_with_warmup(
            worker.optimizer,  # Optimizer to adjust learning rate for
            num_warmup_steps=num_warmup_steps,  # Steps to gradually increase lr
            num_training_steps=num_training_steps  # Total steps for full cycle
        )

```{tip}
* `shuffle=True` 通过打乱数据索引实现随机性，而非加载全部数据后打乱。
* 随机性在每个 epoch 内有效，不同 epoch 打乱结果不同（可通过 `torch.manual_seed` 固定随机性）。
```

## Worker

```{tip}
在 PyTorch 分布式训练中，world_size 表示参与分布式训练的总进程数。如果两台机器各有 8 块 GPU，且每块 GPU 分配一个进程（这是最常见的配置），那么 world_size = 16（2 台机器 × 8 GPU / 台 = 16 个进程）。<br>
`dist.get_world_size()`：返回分布式训练中总进程数量。每个进程通常对应一块 GPU
```